# My Custom Shakespeare LLM: A Step-by-Step Guide

Welcome! This notebook showcases my custom **Generative Pre-trained Transformer (GPT)** built, trained, and evaluated from absolute scratch using PyTorch.

My custom model consists of **10.8 Million parameters** trained on the works of William Shakespeare. It is fully capable of generating unique, Shakespeare-like dialogue using creative sampling techniques!

This notebook breaks down how my custom model is configured and executed:
1. **Data Prep & Tokenization**: Converting raw text characters into integers that my network processes.
2. **Network Architecture**: Building my Transformer model block-by-block (Attention, MLP, LayerNorm).
3. **Training Setup**: Defining my training objective, optimizer, learning rate schedules, and data batching.
4. **The Training Loop**: Interactively training my model, logging loss values, and saving progress checkpoints.
5. **Text Generation (Inference)**: Loading my trained checkpoint and using top-k/temperature sampling to generate creative text.

## Step 1: Data Preparation & Character-Level Tokenization

Computers don't understand words or characters directly; they process numbers. To feed text into a neural network:
1. We download the text dataset (Tiny Shakespeare).
2. We find all unique characters in the text (the vocabulary).
3. We build a dictionary mapping every unique character to a unique index integer (`stoi`), and vice-versa (`itos`).
4. We write `encode` and `decode` helper functions to convert back and forth.

In [ ]:
import urllib.request
import os
import torch

# 1. Download the Tiny Shakespeare Dataset
os.makedirs("../data", exist_ok=True)
data_path = "../data/shakespeare.txt"
if not os.path.isfile(data_path):
    print("Downloading Shakespeare dataset...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        data_path
    )

# Read the raw text
with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()

# 2. Extract Vocabulary (all unique characters)
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Dataset Statistics:\n- Total Characters in text: {len(text):,}\n- Vocabulary Size (unique characters): {vocab_size}")

# 3. Create Mapping Dictionaries
stoi = {ch: i for i, ch in enumerate(chars)}  # character to integer
itos = {i: ch for i, ch in enumerate(chars)}  # integer to character

# 4. Define Encoding and Decoding Helpers
def encode(s: str) -> list[int]:
    # Takes a string and returns a list of matching character integers
    return [stoi[c] for c in s if c in stoi]

def decode(ids: list[int]) -> str:
    # Takes a list of integers and returns a reconstructed string
    return "".join([itos[i] for i in ids])

# Demo
test_sentence = "Hello Shakespeare!"
encoded_test = encode(test_sentence)
print(f"\nTest Sentence: {test_sentence}")
print(f"Encoded: {encoded_test}")
print(f"Decoded: {decode(encoded_test)}")

: 

## Step 2: The Transformer Architecture Block-by-Block

Now, we define the structural blueprint of our model. We will construct:
* **GPTConfig**: A data class holding model shape parameters (number of layers, heads, embedding size).
* **CausalSelfAttention**: The core mechanism where tokens communicate with previous tokens to model relationships.
* **MLP (Multi-Layer Perceptron)**: A fully connected feed-forward layer that processes each token individually.
* **Block**: A single Transformer layer combining attention, MLP, and normalizations with residual pathways.
* **GPT**: The master class orchestrating token/position embeddings, transformer blocks, and output vocabulary projection.

In [ ]:
from dataclasses import dataclass
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class GPTConfig:
    vocab_size: int = 65       # Character level vocabulary limit
    block_size: int = 256      # Maximum context window size
    n_layer: int = 6           # Number of sequential transformer layers
    n_head: int = 6            # Multi-head attention head count
    n_embd: int = 384          # Embedding dimensionality size

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # Query, Key, and Value projections combined in a single linear layer
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # Output projection layer
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.shape
        # Project x to Query, Key, and Value vectors
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # Reshape for multi-head attention processing
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2) # (B, n_head, T, head_dim)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2) # (B, n_head, T, head_dim)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2) # (B, n_head, T, head_dim)

        # Scaled dot-product causal attention
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        # Concatenate heads and project output back to model dimension
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Expand representation 4x to learn complex feature interactions
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Pre-normalization layers to keep gradient flows stable
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        # Combine self-attention & residual connections
        x = x + self.attn(self.ln_1(x))
        # Combine MLP & residual connections
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),   # Token lookup dictionary
            wpe = nn.Embedding(config.block_size, config.n_embd),   # Positional memory table
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        # Linear projection mapping embedding dimensions to output token probabilities
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # Weight-tying optimization: share weights between input embeddings and final output layers
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)

        tok_emb = self.transformer.wte(idx)    # (B, T, n_embd)
        pos_emb = self.transformer.wpe(pos)    # (T, n_embd)
        x = tok_emb + pos_emb                  # Broadcast combined embeddings

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)               # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # Flatten outputs and calculate Cross-Entropy loss
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

## Step 3: Preparing the Training Setup (Batching & Learning Rate Schedules)

To train our model efficiently, we need:
1. **Data Batching**: Feed text subsets (chunks of context `block_size`) in batches so the hardware can compute them in parallel.
2. **Device Routing**: Check if we can run computations on a high-speed GPU (CUDA) or if we must fall back to the CPU.
3. **Cosine Learning Rate Decay with Warmup**: Start with a small learning rate, increase it during training warmups, then gradually decrease it according to a cosine wave. This achieves faster and higher quality convergence.

In [ ]:
import math

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def load_data(filepath, block_size, batch_size, device):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {c: i for i, c in enumerate(chars)}
    itos = {i: c for c, i in stoi.items()}

    # Encode all characters into a long integer tensor
    tokens = torch.tensor([stoi[c] for c in text], dtype=torch.long)

    # Function to dynamically yield random mini-batches
    def get_batch(split_tokens):
        # Pick random starting indices
        ix = torch.randint(len(split_tokens) - block_size - 1, (batch_size,))
        x = torch.stack([split_tokens[i:i + block_size] for i in ix]).to(device)
        # Target sequences are shifted left by 1 token (next-token prediction)
        y = torch.stack([split_tokens[i + 1:i + block_size + 1] for i in ix]).to(device)
        return x, y

    # Split dataset: 90% Training, 10% Validation
    n = int(0.9 * len(tokens))
    get_train = lambda: get_batch(tokens[:n])
    get_val = lambda: get_batch(tokens[n:])
    return get_train, get_val, vocab_size, stoi, itos

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    # 1) Linear warmup phase
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    # 2) Beyond max_steps, stay at min_lr
    if step >= max_steps:
        return min_lr
    # 3) Cosine decay phase
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

## Step 4: The Core Training Loop

This cell handles the actual loop. During training:
- We draw random training chunks.
- We calculate loss (how well the model guesses the next character).
- We backpropagate errors to calculate gradients, clamp gradients to prevent explosive values (`clip_grad_norm_`), and adjust model weights using the AdamW optimizer.

In [ ]:
import json
from tqdm.notebook import tqdm

def train(data_path, max_steps=2000, batch_size=32,
          n_layer=4, n_head=4, n_embd=256, block_size=128):
    # Using scaled-down parameters (4 Layers, 256 Emb, 2000 Steps) for CPU-friendlier execution
    device = get_device()
    print(f"Training running on device: {device}")

    get_train_batch, get_val_batch, vocab_size, stoi, itos = load_data(
        data_path, block_size, batch_size, device
    )

    config = GPTConfig(
        vocab_size=vocab_size,
        block_size=block_size,
        n_layer=n_layer,
        n_head=n_head,
        n_embd=n_embd,
    )
    model = GPT(config).to(device)
    print(f"Model parameters size: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

    max_lr = 1e-3
    min_lr = 1e-4
    warmup_steps = 50
    loss_log = {"steps": [], "train": [], "val": []}

    pbar = tqdm(range(max_steps), desc="Training Progress")
    for step in pbar:
        # --- Validation loss estimation ---
        if step % 100 == 0:
            model.eval()
            with torch.no_grad():
                val_losses = []
                for _ in range(10):
                    x, y = get_val_batch()
                    _, loss = model(x, y)
                    val_losses.append(loss.item())
                val_loss = sum(val_losses) / len(val_losses)
                tqdm.write(f"Step {step:4d} | val loss: {val_loss:.4f}")
            model.train()

        # --- Update Learning Rate based on step ---
        lr = get_lr(step, warmup_steps, max_steps, max_lr, min_lr)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        # --- Forward / Backward Training Step ---
        x, y = get_train_batch()
        _, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        # --- Save Logs & Checkpoint ---
        loss_log["steps"].append(step)
        loss_log["train"].append(loss.item())
        if step % 100 == 0:
            loss_log["val"].append(val_loss)

        if step > 0 and step % 1000 == 0:
            torch.save({
                "step": step,
                "model_state_dict": model.state_dict(),
                "config": config,
                "stoi": stoi,
                "itos": itos,
            }, f"checkpoint_{step}.pt")

    # Save Final Checkpoint
    torch.save({
        "step": max_steps,
        "model_state_dict": model.state_dict(),
        "config": config,
        "stoi": stoi,
        "itos": itos,
    }, "checkpoint_final.pt")

    with open("loss_log.json", "w") as f:
        json.dump(loss_log, f)

    print("\nTraining complete! Final checkpoint saved as 'checkpoint_final.pt'.")
    return model, stoi, itos

## Step 5: Advanced Generation Loop (Top-K Sampling & Temperature Controls)

To make our generated Shakespeare dialogue sound natural rather than repetitive:
1. **Temperature**: Scale the output probability logits. Lower temperature (e.g. 0.2) makes the model conservative/predictable; higher temperature (e.g. 1.2) makes it highly creative/unstable.
2. **Top-K Filtering**: Keep only the top-K highest-probability characters and ignore low-probability noise.

In [ ]:
@torch.no_grad()
def generate(model, prompt: str, stoi: dict, itos: dict, max_new_tokens=200, temperature=0.8, top_k=40):
    device = next(model.parameters()).device
    tokens = [stoi[c] for c in prompt if c in stoi]
    idx = torch.tensor([tokens], dtype=torch.long, device=device)

    model.eval()
    for _ in range(max_new_tokens):
        # Keep sequence limited to max supported context window (block_size)
        idx_cond = idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        # Look at logits of the last token, scaled by temperature
        logits = logits[:, -1, :] / temperature

        # Apply top-k filtering
        if top_k > 0:
            values, _ = torch.topk(logits, top_k)
            logits[logits < values[:, -1:]] = float("-inf")

        # Convert logits to probabilities
        probs = torch.softmax(logits, dim=-1)
        # Sample the next token randomly according to the probabilities
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)

    return "".join([itos[i] for i in idx[0].tolist()])

## Step 6: Load Model & Try Generative Inference Loop

In [ ]:
# Execute this block to train model or load final checkpoints
if __name__ == "__main__":
    # Uncomment the line below to start active training (will take a few minutes on CPU, seconds on GPU)
    # model, stoi, itos = train(data_path, max_steps=500)

    checkpoint_path = "checkpoint_final.pt"
    if os.path.exists(checkpoint_path):
        print("Loading trained model checkpoint...")
        # Set weights_only=False to allow PyTorch 2.6+ to load custom GPTConfig objects
        checkpoint = torch.load(checkpoint_path, map_location=get_device(), weights_only=False)
        config = checkpoint['config']
        stoi = checkpoint['stoi']
        itos = checkpoint['itos']

        loaded_model = GPT(config).to(get_device())
        loaded_model.load_state_dict(checkpoint['model_state_dict'])

        prompt = "ROMEO:\nThou art"
        print(f"\nPrompt: '{prompt}'")
        generated_sample = generate(loaded_model, prompt, stoi, itos, max_new_tokens=150, temperature=0.85, top_k=30)
        print("\n--- Generated Text Output ---")
        print(generated_sample)
    else:
        print("No trained model checkpoint found yet. Run `train(data_path)` first!")

### Quick-Start Real-time Training (Educational Mode)

Run the cell below to perform a rapid training run on your dataset. This establishes a fully initialized, functional checkpoint (`checkpoint_final.pt`) in less than a minute, enabling you to test the generation and evaluation steps immediately!

In [ ]:
# Train a lightweight version (4 layers, 4 attention heads, 128 embedding dimension)
# to quickly demonstrate the training pipeline and save an operational checkpoint.
model, stoi, itos = train(
    data_path="../data/shakespeare.txt",
    max_steps=150,
    batch_size=16,
    n_layer=4,
    n_head=4,
    n_embd=128,
    block_size=64
)

In [ ]:
!pip install torch numpy tqdm tiktoken


## Part 1 : Tokenization

In [ ]:
import urllib.request, os
os.makedirs("../data", exist_ok=True)
if not os.path.isfile("../data/shakespeare.txt"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", "../data/shakespeare.txt")

In [ ]:
text = open("../data/shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)  # 65 for Shakespeare

stoi = {c: i for i, c in enumerate(chars)}  # string to int
itos = {i: c for c, i in stoi.items()}      # int to string

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join([itos[i] for i in ids])

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode("Hello, world!")  # [15496, 11, 995, 0]
text = enc.decode(tokens)             # "Hello, world!"

## Part 2 : The Transformer


In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size: int = 65       # character-level: 65 unique chars in Shakespeare
    block_size: int = 256      # max sequence length (context window)
    n_layer: int = 6           # number of transformer blocks
    n_head: int = 6            # number of attention heads
    n_embd: int = 384          # embedding dimension

In [ ]:
import torch
import torch.nn as nn

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),   # token embeddings
            wpe = nn.Embedding(config.block_size, config.n_embd),   # position embeddings
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # weight tying: the output projection shares weights with the token embeddings
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)

        tok_emb = self.transformer.wte(idx)    # (B, T, n_embd)
        pos_emb = self.transformer.wpe(pos)    # (T, n_embd)
        x = tok_emb + pos_emb                  # (B, T, n_embd) — broadcasting adds position info

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)               # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss

In [ ]:
# Cell cleared since the forward function is now correctly defined inside the GPT class in cell EojcONp7R-1f.

token IDs (B, T)
    │
    ▼
┌─────────┐     ┌─────────┐
│   wte   │     │   wpe   │
│ [65,384]│     │[256,384]│
└─────────┘     └─────────┘
    │               │
    ▼               ▼
  tok_emb    +   pos_emb      → x (B, T, 384)
                                  │
                                  ▼
                          ┌──────────────┐
                          │ Block × 6    │
                          └──────────────┘
                                  │
                                  ▼
                          ┌──────────────┐
                          │  LayerNorm   │
                          │   lm_head    │  Linear: 384 → 65
                          └──────────────┘
                                  │
                                  ▼
                          logits (B, T, 65)

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)  # Q, K, V projections
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)       # output projection
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # reshape for multi-head: (B, T, C) → (B, n_head, T, head_dim)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        # attention with causal mask (each token can only attend to previous tokens)
        y = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, is_causal=True
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

In [ ]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        x = self.c_fc(x)       # project up: 384 → 1536
        x = self.gelu(x)       # non-linearity
        return self.c_proj(x)  # project back down: 1536 → 384

In [ ]:
##Transformer Block

In [ ]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # attention with residual connection
        x = x + self.mlp(self.ln_2(x))    # MLP with residual connection
        return x

In [ ]:
config = GPTConfig()
model = GPT(config)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params / 1e6:.1f}M")  # ~10.8M

## Part 3 : The Training Loop

###Step 1 : Data Loading (Character-Level)

In [ ]:
import torch

def load_data(filepath, block_size, batch_size, device):
    with open(filepath, "r") as f:
        text = f.read()

    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {c: i for i, c in enumerate(chars)}
    itos = {i: c for c, i in stoi.items()}

    tokens = torch.tensor([stoi[c] for c in text], dtype=torch.long)
    print(f"Dataset: {len(tokens):,} chars, vocab size: {vocab_size}")

    def get_batch(split_tokens):
        ix = torch.randint(len(split_tokens) - block_size - 1, (batch_size,))
        x = torch.stack([split_tokens[i:i + block_size] for i in ix]).to(device)
        y = torch.stack([split_tokens[i + 1:i + block_size + 1] for i in ix]).to(device)
        return x, y

    n = int(0.9 * len(tokens))
    get_train = lambda: get_batch(tokens[:n])
    get_val = lambda: get_batch(tokens[n:])
    return get_train, get_val, vocab_size, stoi, itos

###Step 2: Device Setup

In [ ]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")     # Apple Silicon GPU
    elif torch.cuda.is_available():
        return torch.device("cuda")    # NVIDIA GPU
    return torch.device("cpu")

### Step 3: Learning Rate Schedule

In [ ]:
import math

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

###Step 4: The Optimizer

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

###Step 5: The Full Training Loop

In [ ]:
import json
from tqdm import tqdm

def train(data_path, max_steps=5000, batch_size=64,
          n_layer=6, n_head=6, n_embd=384, block_size=256):
    device = get_device()
    print(f"Using device: {device}")

    get_train_batch, get_val_batch, vocab_size, stoi, itos = load_data(
        data_path, block_size, batch_size, device
    )

    config = GPTConfig(
        vocab_size=vocab_size,
        block_size=block_size,
        n_layer=n_layer,
        n_head=n_head,
        n_embd=n_embd,
    )
    model = GPT(config).to(device)
    print(f"Model: {n_layer}L/{n_head}H/{n_embd}D, "
          f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

    max_lr = 1e-3
    min_lr = max_lr * 0.1
    warmup_steps = 100

    loss_log = {"steps": [], "train": [], "val": []}

    pbar = tqdm(range(max_steps), desc="Training")
    for step in pbar:
        # --- validation loss ---
        if step % 100 == 0:
            model.eval()
            with torch.no_grad():
                val_losses = []
                for _ in range(20):
                    x, y = get_val_batch()
                    _, loss = model(x, y)
                    val_losses.append(loss.item())
                val_loss = sum(val_losses) / len(val_losses)
                tqdm.write(f"Step {step:5d} | val loss: {val_loss:.4f}")
            model.train()

        # --- update learning rate ---
        lr = get_lr(step, warmup_steps, max_steps, max_lr, min_lr)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        # --- training step ---
        x, y = get_train_batch()
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        # --- log loss ---
        loss_log["steps"].append(step)
        loss_log["train"].append(loss.item())
        if step % 100 == 0:
            loss_log["val"].append(val_loss)

        # --- generate sample ---
        if step > 0 and step % 100 == 0:
            model.eval()
            sample = generate(model, "To be or not", stoi, itos,
                            max_new_tokens=100, temperature=0.8)
            tqdm.write(f"\n--- Step {step} sample ---\n{sample}\n---\n")
            model.train()

        # --- save checkpoint ---
        if step > 0 and step % 1000 == 0:
            torch.save({
                "step": step,
                "model_state_dict": model.state_dict(),
                "config": config,
                "stoi": stoi,
                "itos": itos,
            }, f"checkpoint_{step}.pt")

    # --- save final checkpoint and loss log ---
    torch.save({
        "step": max_steps,
        "model_state_dict": model.state_dict(),
        "config": config,
        "stoi": stoi,
        "itos": itos,
    }, "checkpoint_final.pt")

    with open("loss_log.json", "w") as f:
        json.dump(loss_log, f)

    return model, stoi, itos

###Step 6: Entry Point

In [ ]:
if __name__ == "__main__":
    train("../data/shakespeare.txt")

In [ ]:
if __name__ == "__main__":
    import sys
    data_path = sys.argv[1] if len(sys.argv) > 1 else "../data/shakespeare.txt"
    train(data_path)

## Part 4 : Text Generation


###The Naive Approach: Greedy Decoding

In [ ]:
def generate_greedy(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]
        next_token = logits.argmax(dim=-1, keepdim=True)
        idx = torch.cat([idx, next_token], dim=1)
    return idx

###Temperature

In [ ]:
# Cleaned up redundant statement: logits / temperature is already integrated inside the generate function.

###Top-k Sampling


In [ ]:
# Cleaned up redundant statement: top-k logic is already fully defined within the generate function.

The Full Generate Function

In [ ]:
@torch.no_grad()
def generate(model, prompt, stoi, itos, max_new_tokens=200, temperature=0.8, top_k=40):
    device = next(model.parameters()).device
    tokens = [stoi[c] for c in prompt if c in stoi]
    idx = torch.tensor([tokens], dtype=torch.long, device=device)

    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        if top_k > 0:
            values, _ = torch.topk(logits, top_k)
            logits[logits < values[:, -1:]] = float("-inf")

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)

    return "".join([itos[i] for i in idx[0].tolist()])

### Part 5: Putting It All Together (Model Loading & Generation Demonstration)

Now, we load the final model from the checkpoint we trained in the previous steps and generate text using our top-k sampling strategy.

In [ ]:
import torch
import os

def load_model_and_generate(checkpoint_path, prompt="ROMEO:", max_tokens=500, temperature=0.8, top_k=40):
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint '{checkpoint_path}' not found! Please run the training loop cell first.")
        return

    print(f"Loading checkpoint from {checkpoint_path}...")
    # Set weights_only=False to allow PyTorch 2.6+ to load custom GPTConfig objects
    checkpoint = torch.load(checkpoint_path, map_location=get_device(), weights_only=False)

    # Retrieve configurations and vocab dictionaries
    config = checkpoint['config']
    stoi = checkpoint['stoi']
    itos = checkpoint['itos']

    # Initialize and configure the model
    model = GPT(config)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(get_device())
    model.eval()

    print("Generating text...")
    generated_text = generate(model, prompt, stoi, itos, max_new_tokens=max_tokens, temperature=temperature, top_k=top_k)
    print("\n--- Generated Output ---\n")
    print(generated_text)
    print("\n------------------------\n")

# Let's run a test on the final checkpoint if available
load_model_and_generate("checkpoint_final.pt", prompt="ROMEO:\nShall I speak more, or", max_tokens=300)

### Part 6: Text Generation Demonstration Loop

Run the following cell to compare how different prompts, temperatures, and `top_k` settings impact the diversity and style of the model's generated text.

In [ ]:
# Load the model once so we can quickly run the loop
checkpoint_path = "checkpoint_final.pt"
if not os.path.exists(checkpoint_path):
    print(f"Checkpoint '{checkpoint_path}' not found! Please make sure you have trained the model and saved the checkpoint.")
else:
    # Set weights_only=False to allow PyTorch 2.6+ to load custom GPTConfig objects
    checkpoint = torch.load(checkpoint_path, map_location=get_device(), weights_only=False)
    config = checkpoint['config']
    stoi = checkpoint['stoi']
    itos = checkpoint['itos']

    demo_model = GPT(config)
    demo_model.load_state_dict(checkpoint['model_state_dict'])
    demo_model.to(get_device())
    demo_model.eval()

    # Configuration sets for the demonstration loop
    test_prompts = [
        "ROMEO:",
        "JULIET:\nO Romeo, Romeo!",
        "To be, or not to be,"
    ]

    # Generate texts demonstrating various temperature/creativity profiles
    for idx, prompt in enumerate(test_prompts):
        print(f"=== Demo {idx+1}: Prompt -> '{prompt.replace('\n', ' ')}' ===")
        for temp, tk in [(0.5, 10), (0.8, 40), (1.2, 60)]:
            print(f"\n[Config: Temp={temp}, Top-k={tk}]")
            out = generate(demo_model, prompt, stoi, itos, max_new_tokens=100, temperature=temp, top_k=tk)
            print(out)
            print("-" * 50)
        print("\n" + "="*60 + "\n")